In [1]:
#Imports
import sys
import os

module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)


import pandas as pd
# see https://github.com/marcharper/python-ternary for installation
import numpy as np
from scipy.stats import multinomial, dirichlet
from scipy.linalg import block_diag

import random

from ast import literal_eval
from src.model_funcs import run_em, tolerance_sort, max_posterior_label, sort_matrix, MultinomialExpectationMaximizer
from src.plotting_funcs import  plot_ternary_axes, plot_ternary_bounds, CB_color_cycle

# Bayesian Mixture Model

**Goal**: Model the uncertainty expressed by multiple annotations
**Tool**: Multinomial Mixture Model
- There exists a latent ground truth, i.e. a "true label" for each image
- This latent ground truth is unambiguous but the annotator’s opinion about it is not
- We set up a distributional framework and model the observed annotations under the assumption that there exists a latent true label
- The parameters of the associated distributions allow us to analyse the sources of labelling uncertainty and we can estimate the latent labels through the posterior probabilities

Assume latent ground truth:

$Z^{(i)}\sim Multi(\boldsymbol{\pi}, 1)$ iid.  $\rightarrow$ prior

Given latent true class, the labellers' votes follow:

$Y^{(i)}|Z^{(i)}\sim Multi(\boldsymbol{\theta_{Z^{(i)}}}, J^i)$ iid. $\rightarrow$ likelihood

or explicitly:

$Y^{(i)}|(Z^{(i)} = airplane) \sim Multi(\boldsymbol{\theta_{airplane}}, J^i) \\
Y^{(i)}|(Z^{(i)} = automobile) \sim Multi(\boldsymbol{\theta_{automobile}}, J^i)\\
Y^{(i)}|(Z^{(i)} = bird) \sim Multi(\boldsymbol{\theta_{bird}}, J^i)\\
Y^{(i)}|(Z^{(i)} = cat) \sim Multi(\boldsymbol{\theta_{cat}}, J^i)\\
Y^{(i)}|(Z^{(i)} = deer) \sim Multi(\boldsymbol{\theta_{deer}}, J^i)\\
Y^{(i)}|(Z^{(i)} = dog) \sim Multi(\boldsymbol{\theta_{dog}}, J^i)\\
Y^{(i)}|(Z^{(i)} = frog) \sim Multi(\boldsymbol{\theta_{frog}}, J^i)\\
Y^{(i)}|(Z^{(i)} = horse) \sim Multi(\boldsymbol{\theta_{horse}}, J^i)\\
Y^{(i)}|(Z^{(i)} = ship) \sim Multi(\boldsymbol{\theta_{ship}}, J^i)\\
Y^{(i)}|(Z^{(i)} = truck) \sim Multi(\boldsymbol{\theta_{truck}}, J^i).$


Use Bayes Rule to model latent ground truth given votes:

$\tau^{(i)}_l = P(Z^{(i)}=l|Y^{(i)}) = \frac{P(Z^{(i)}=l) \cdot P(Y^{(i)}|Z^{(i)} = l)}{P(Y^{(i)})} = \frac{prior \cdot likelihood}{evidence}$



Apply Expectation Maximization (EM) algorithm, iteratively estimate latent ground truth and parameters

1. E-Step:
calculate posterior class probabilities $\tau^{(i)}_l$ given $\pi$ and $\theta_{Z^{(i)}}$, i.e., parameters of prior and likelihood
choose class with highest posterior as latent ground truth class $Z^{(i)}$

2. M-Step:
update $\pi$ and $\theta_{Z^{(i)}}$ given $\tau^{(i)}_l$, i.e.,  posterior class probabilities

Initial values:
- $\pi$ = (1/10, 1/10, 1/10, 1/10, 1/10, 1/10, 1/10, 1/10, 1/10, 1/10)
- $\theta$ drawn from dirichlet with $\alpha$ = (20,20,20,20,20,20,20,20,20,20) (i.e., 2* number_of_observed_classes)

# 1. Run EM

In [2]:
# load data
df_cifar10h = pd.read_csv('../cifar-10h/data/Y_cifar10h.csv', index_col=0)
# df_R_cifar10h = pd.read_csv('../cifar-10h/data/Y_R_cifar10h.csv', index_col=1)
# df_snli.old_labels = df_snli.old_labels.apply(literal_eval) # since quotes in list elements are escaped
# df_cifar10h["J"].max()


In [3]:
# df_R_cifar10h[df_R_cifar10h["Reaction"] == "Fast"]

In [3]:
#extract relevant columns
cifar10h_one_hot = df_cifar10h[['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']]
cifar10h_one_hot_arr = np.array(cifar10h_one_hot).astype(int)

# filter df_R_cifar10h for Reaction == Slow
# Slow_cifar10h_one_hot = df_R_cifar10h[df_R_cifar10h['Reaction'] == 'Slow'][['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']]
# Slow_cifar10h_one_hot_arr = np.array(Slow_cifar10h_one_hot).astype(int)
# Fast_cifar10h_one_hot = df_R_cifar10h[df_R_cifar10h['Reaction'] == 'Fast'][['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']]
# Fast_cifar10h_one_hot_arr = np.array(Fast_cifar10h_one_hot).astype(int)
# # cifar10h_one_hot_arr_test = cifar10h_one_hot_arr[0:100]
# #frequency of all votes
# rel_freq = np.sum(cifar10h_one_hot_arr,axis=0)/(100*len(cifar10h_one_hot_arr))

#frequency of personal ground truth
#gt_freq = np.unique(df_cifar10h['ground_truth'],return_counts=True)
#as the labels are initially sorted alphabetically, we manually fix the order: E, N, C
#rel_freq_gt = gt_freq[1][[1,2,0]]/len(df_snli)

#frequency of majority vote label
#m_vote_freq = np.unique(df_snli['majority_label'], return_counts=True)
#as the labels are initially sorted alphabetically, we manually fix the order: E, N, C
#rel_freq_m_vote = m_vote_freq[1][[1,2,0]]/len(df_snli)

In [50]:
likelihoods = []
best_pi = None
best_theta = None
best_tau = None
best_theta_list = None
best_Z_list = None

model = MultinomialExpectationMaximizer(10, restarts=0)

tau, Z = model._e_step(cifar10h_one_hot_arr, np.array([1 / model._K] * model._K), dirichlet.rvs([2 * 10] * 10, model._K))

pi, theta = model._m_step(cifar10h_one_hot_arr, tau, Z)



# np.array([np.mean(Z == k) for k in range(1, 10 + 1)])
# np.mean(Z == 2)

# np.array([multinomial.rvs(1, tau_i) for tau_i in tau])
# cifar10h_one_hot_arr[1:10]

In [52]:
tau_2, Z_2 = model._e_step(cifar10h_one_hot_arr, pi, theta)
pi_2, theta_2 = model._m_step(cifar10h_one_hot_arr, tau_2, Z_2)

7935

In [93]:
T = np.zeros((10, 10))
for l in range(1, 11):
    mask = (Z == l)
    numerator = np.sum(cifar10h_one_hot_arr[mask], axis=0)
    denominator = np.sum(cifar10h_one_hot_arr[mask])
    if denominator != 0:
        T[l-1, :] = numerator / denominator

T

array([[1.88856277e-03, 5.00348843e-01, 7.01294327e-03, 4.57044219e-01,
        2.42987057e-03, 1.84886686e-02, 3.83727085e-03, 1.61189434e-03,
        1.53971996e-03, 5.79800799e-03],
       [1.09769484e-03, 7.69932438e-02, 2.00986379e-03, 5.33386930e-03,
        8.22498106e-03, 1.20050710e-01, 9.12168952e-04, 7.79440639e-01,
        8.03945517e-04, 5.13288292e-03],
       [1.51236902e-03, 3.78092255e-04, 1.31612113e-02, 1.35573080e-02,
        5.05923445e-03, 7.41780995e-02, 8.88678838e-01, 1.40434266e-03,
        1.42234705e-03, 6.48158151e-04],
       [1.02827763e-03, 9.25449871e-04, 5.65552699e-03, 4.61696658e-02,
        4.83290488e-03, 9.15167095e-01, 6.37532134e-03, 3.29048843e-03,
        1.62467866e-02, 3.08483290e-04],
       [1.92325589e-03, 2.60570152e-03, 9.08893507e-03, 9.86444148e-03,
        9.01262524e-01, 1.02366846e-02, 4.12569408e-03, 5.48127928e-02,
        5.05630177e-03, 1.02366846e-03],
       [5.18871550e-03, 2.89222846e-02, 1.20109155e-03, 1.47974479e-03,
   

In [81]:
#np.sum((Z == 2) * cifar10h_one_hot_arr[:, 2-1].reshape(-1, 1))
len(cifar10h_one_hot_arr)

10000

In [8]:
# np.array([np.sum(Z_2 == k) for k in range(1, 10 + 1)])
# np.array([np.sum(Z == k) for k in range(1, 10 + 1)])

theta_2

array([[4.71697930e-01, 4.36613281e-03, 7.11806062e-03, 2.08096308e-03,
        5.62054513e-03, 3.79240935e-03, 1.61420500e-03, 4.91943561e-01,
        9.31571322e-03, 2.45047989e-03],
       [2.62621112e-01, 0.00000000e+00, 0.00000000e+00, 9.17899031e-03,
        0.00000000e+00, 9.68893422e-03, 7.13921469e-03, 2.54971953e-03,
        7.08822030e-01, 0.00000000e+00],
       [1.80487900e-03, 2.43755701e-02, 7.18070138e-04, 1.08680886e-03,
        7.95699342e-04, 1.02858695e-03, 7.18070138e-04, 1.24206727e-03,
        3.78442370e-03, 9.64445825e-01],
       [1.02085460e-03, 1.75003646e-03, 9.05643868e-02, 1.45398862e-01,
        2.20212921e-02, 2.42088377e-02, 7.05993875e-01, 3.35423655e-03,
        4.52092752e-03, 1.16669097e-03],
       [0.00000000e+00, 6.17919670e-04, 7.41503605e-03, 1.64778579e-02,
        4.53141092e-03, 3.58393409e-02, 9.32646756e-01, 1.02986612e-03,
        4.11946447e-04, 1.02986612e-03],
       [1.28810648e-03, 8.75912409e-04, 1.15757836e-02, 8.23701159e-01,
   

In [24]:
# pi.sum()

# theta
# Z[20:45] == 3
# dirichlet.rvs([2 * 10] * 10, model._K)
# sum(theta[5,:])

Z = np.argmax(np.array([multinomial.rvs(1, tau_i) for tau_i in cifar10h_em[3]]), axis=1) + 1

np.sum((Z == 2) * cifar10h_one_hot_arr[:, 3-1].reshape(-1, 1))/np.sum((Z == 2) * cifar10h_one_hot_arr.sum(axis=1).reshape(-1, 1))
# loss = 0
# for k in range(pi.shape[0]):
#     weights = tau[:, k]
#     loss += np.sum(weights * (np.log(pi[k]) + np.log(model._multinomial_prob(cifar10h_one_hot_arr, theta[k]))))
#     loss -= np.sum(weights * np.log(weights))

0.10047063399455465

In [17]:
# run em
# random.seed(12)
df_cifar10h = pd.read_csv('../cifar-10h/data/Y_cifar10h.csv', index_col=0)
cifar10h_one_hot = df_cifar10h[['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']]
cifar10h_one_hot_arr = np.array(cifar10h_one_hot).astype(int)
cifar10h_em = run_em(cifar10h_one_hot_arr, K=10, max_iter=101, restarts=1, seed=13)
# cifar10h_em_slow = run_em(Slow_cifar10h_one_hot_arr, K=10)
# cifar10h_em_fast = run_em(Fast_cifar10h_one_hot_arr, K=10)

In [18]:
# how many iterations did it do. look at the length of the list
cifar10h_em[2]
theta_sorted = sort_matrix(cifar10h_em[2])
theta_sorted


array([[9.47376585e-01, 8.14363827e-03, 1.15756001e-02, 2.32675379e-03,
        1.31849381e-03, 1.27971458e-03, 2.77271493e-03, 1.33788343e-03,
        2.02427580e-02, 3.62585799e-03],
       [1.85138269e-03, 9.71020989e-01, 6.82088360e-04, 1.96831212e-03,
        1.05236490e-03, 9.54923704e-04, 6.43111882e-04, 5.26182449e-04,
        1.03287666e-03, 2.02677684e-02],
       [3.45258066e-03, 1.01431748e-03, 9.46709320e-01, 1.10599618e-02,
        9.20688175e-03, 8.89478407e-03, 1.28350174e-02, 3.74517224e-03,
        2.32122654e-03, 7.60738111e-04],
       [1.32494203e-03, 9.93706525e-04, 1.42431269e-02, 9.17346998e-01,
        6.13759913e-03, 4.21838162e-02, 1.09892251e-02, 3.35132397e-03,
        1.90947528e-03, 1.51978645e-03],
       [1.58804846e-03, 9.60671294e-04, 1.17241109e-02, 9.82237384e-03,
        9.08971494e-01, 1.98800141e-02, 7.64615928e-03, 3.66623534e-02,
        1.60765400e-03, 1.13712112e-03],
       [7.01549255e-04, 4.87186982e-04, 5.02776966e-03, 3.04589301e-02,
   

In [7]:
theta_sorted = sort_matrix(cifar10h_em[2])
# theta_sorted_slow = sort_matrix(cifar10h_em_slow[2])
# theta_sorted_fast = sort_matrix(cifar10h_em_fast[2])

In [8]:
df_cifar10h['posterior'] = cifar10h_em[3].tolist()

In [10]:
theta_sorted

array([[9.47376585e-01, 8.14363827e-03, 1.15756001e-02, 2.32675379e-03,
        1.31849381e-03, 1.27971458e-03, 2.77271493e-03, 1.33788343e-03,
        2.02427580e-02, 3.62585799e-03],
       [1.85138269e-03, 9.71020989e-01, 6.82088360e-04, 1.96831212e-03,
        1.05236490e-03, 9.54923704e-04, 6.43111882e-04, 5.26182449e-04,
        1.03287666e-03, 2.02677684e-02],
       [3.48117646e-03, 1.01242370e-03, 9.45679627e-01, 1.10568332e-02,
        9.91003558e-03, 8.91516769e-03, 1.30777832e-02, 3.77322175e-03,
        2.33441366e-03, 7.59317772e-04],
       [1.32507900e-03, 9.93809249e-04, 1.42426488e-02, 9.17395018e-01,
        6.13433268e-03, 4.21374650e-02, 1.09903611e-02, 3.35167041e-03,
        1.90967267e-03, 1.51994356e-03],
       [1.55568005e-03, 9.62481009e-04, 1.10017100e-02, 9.82319726e-03,
        9.09957010e-01, 1.98801417e-02, 7.39145599e-03, 3.66960581e-02,
        1.59300251e-03, 1.13926324e-03],
       [7.01476871e-04, 4.87136716e-04, 5.02919764e-03, 3.05025092e-02,
   

In [43]:
theta_sorted_slow

array([[8.92582655e-01, 1.07839174e-02, 3.01286691e-02, 4.90971303e-03,
        1.85845445e-03, 3.19661084e-03, 6.92148485e-03, 2.38029956e-03,
        3.97992205e-02, 7.43897563e-03],
       [2.50224266e-03, 9.26944371e-01, 9.52985546e-04, 4.04088769e-03,
        5.95617529e-04, 4.76489600e-04, 9.57347974e-04, 2.38407495e-04,
        1.31011902e-03, 6.19815311e-02],
       [6.29314898e-03, 1.82559340e-03, 8.80562944e-01, 2.54726761e-02,
        2.33218920e-02, 2.30542259e-02, 2.47047301e-02, 9.50882900e-03,
        4.20275794e-03, 1.05320242e-03],
       [1.94674792e-03, 2.24858916e-03, 3.43960511e-02, 8.17014185e-01,
        1.43958807e-02, 9.24988590e-02, 2.34244898e-02, 7.78263144e-03,
        4.15127214e-03, 2.14129399e-03],
       [2.13046873e-03, 1.21158583e-03, 2.26754321e-02, 1.86541527e-02,
        8.28621211e-01, 4.56602004e-02, 1.15293640e-02, 6.61202350e-02,
        2.06442356e-03, 1.33292688e-03],
       [6.82785577e-04, 2.53838083e-04, 1.01858284e-02, 6.07730565e-02,
   

In [44]:
theta_sorted_fast

array([[9.67478995e-01, 7.19433375e-03, 4.89579634e-03, 1.47037261e-03,
        1.11613121e-03, 6.56416343e-04, 1.31283269e-03, 9.71496183e-04,
        1.27768367e-02, 2.12678909e-03],
       [1.74196354e-03, 9.78842531e-01, 6.27106874e-04, 1.20755857e-03,
        1.13808284e-03, 9.05821039e-04, 5.80654537e-04, 5.80654512e-04,
        9.29047244e-04, 1.34465801e-02],
       [2.41819088e-03, 6.75463678e-04, 9.72471688e-01, 5.03952060e-03,
        4.48433908e-03, 3.62047162e-03, 7.39853414e-03, 1.62114390e-03,
        1.62217190e-03, 6.48476189e-04],
       [1.07553096e-03, 7.17028591e-04, 5.99127381e-03, 9.56449492e-01,
        3.12439291e-03, 2.24590246e-02, 6.32688527e-03, 1.59927465e-03,
        9.88001852e-04, 1.26909518e-03],
       [1.25262816e-03, 8.15748164e-04, 5.37884392e-03, 4.36926932e-03,
        9.50674959e-01, 7.58270796e-03, 4.95242908e-03, 2.25560049e-02,
        1.36920285e-03, 1.04820656e-03],
       [6.84994286e-04, 5.53264605e-04, 2.78037443e-03, 1.76957722e-02,
   

In [19]:
# Convert theta_matched, pi_matched, and tau_matched to DataFrames
df_theta = pd.DataFrame(cifar10h_em[2])
df_pi = pd.DataFrame(cifar10h_em[1])
df_tau = pd.DataFrame(cifar10h_em[3])

# df_theta_slow = pd.DataFrame(cifar10h_em_slow[2])
# df_pi_slow = pd.DataFrame(cifar10h_em_slow[1])
# df_tau_slow = pd.DataFrame(cifar10h_em_slow[3])

# df_theta_fast = pd.DataFrame(cifar10h_em_fast[2])
# df_pi_fast = pd.DataFrame(cifar10h_em_fast[1])
# df_tau_fast = pd.DataFrame(cifar10h_em_fast[3])

# Export DataFrames to CSV files
df_theta.to_csv('../cifar-10h/data/theta.csv', index=False)
df_pi.to_csv('../cifar-10h/data/pi.csv', index=False)
df_tau.to_csv('../cifar-10h/data/tau.csv', index=False)

# df_theta_slow.to_csv('../cifar-10h/data/theta_slow.csv', index=False)
# df_pi_slow.to_csv('../cifar-10h/data/pi_slow.csv', index=False)
# df_tau_slow.to_csv('../cifar-10h/data/tau_slow.csv', index=False)

# df_theta_fast.to_csv('../cifar-10h/data/theta_fast.csv', index=False)
# df_pi_fast.to_csv('../cifar-10h/data/pi_fast.csv', index=False)
# df_tau_fast.to_csv('../cifar-10h/data/tau_fast.csv', index=False)

In [45]:
# We define theta_cut as theta without the last column and written as a single array row by row
theta_old = theta_sorted[:, :-1].flatten()

def theta_old(theta):
    theta_sorted = sort_matrix(theta)
    return theta_sorted[:, :-1].flatten()

# apply the function to all elements of cifar10h_em[4]
theta_old_list = list(map(theta_old, cifar10h_em[4][100:]))
theta_old_final = np.mean(theta_old_list, axis=0)

def compute_operation(vec, mean_vec):
    diff = vec - mean_vec
    return np.outer(diff, diff)

# Apply the function to each element in the list using list comprehension
result = [compute_operation(vec, theta_old_final) for vec in theta_old_list]

# take the mean of the resulting matrices
combined_result = np.mean(result, axis=0)


def var_theta(theta, Z):
    theta_sorted = sort_matrix(theta)

    return block_diag(*[(np.diag(theta_sorted[:, :-1][i-1]) - np.outer(theta_sorted[:, :-1][i-1], theta_sorted[:, :-1][i-1])) / np.sum(Z == i) for i in range(1, 11)])

# apply the function to all elements of cifar10h_em[4] and cifar10h_em[7]
var_theta_list = list(map(var_theta, cifar10h_em[4][100:], cifar10h_em[7][100:]))
var_theta_final = np.mean(var_theta_list, axis=0)


var_theta_old_estimated = var_theta_final + combined_result


In [51]:
var_theta_old_estimated

array([[ 4.97050759e-05, -7.69201616e-06, -1.09336515e-05, ...,
         1.41820666e-31,  1.40785479e-31,  5.05171424e-31],
       [-7.69201616e-06,  8.05315994e-06, -9.39855434e-08, ...,
        -1.72637802e-33, -1.71377672e-33, -6.14943413e-33],
       [-1.09336515e-05, -9.39855434e-08,  1.14073834e-05, ...,
        -2.44784944e-33, -2.42998192e-33, -8.71934690e-33],
       ...,
       [ 1.41820666e-31, -1.72637802e-33, -2.44784944e-33, ...,
         7.14016998e-07, -8.88391615e-10, -2.70681820e-09],
       [ 1.40785479e-31, -1.71377672e-33, -2.42998192e-33, ...,
        -8.88391615e-10,  1.23440814e-06, -4.68206392e-09],
       [ 5.05171424e-31, -6.14943413e-33, -8.71934690e-33, ...,
        -2.70681820e-09, -4.68206392e-09,  3.75150371e-06]])

In [4]:
np.array([np.array([1 / 10] * 10)] * 10)

array([[0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1],
       [0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1],
       [0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1],
       [0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1],
       [0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1],
       [0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1],
       [0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1],
       [0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1],
       [0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1],
       [0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1]])

In [13]:
np.log(0)

/var/folders/2t/c8n8t7mj7xv62mh_p5b0h93w0000gn/T/ipykernel_4051/2545229922.py:1: RuntimeWarning: divide by zero encountered in log
  0 * np.log(0)
/var/folders/2t/c8n8t7mj7xv62mh_p5b0h93w0000gn/T/ipykernel_4051/2545229922.py:1: RuntimeWarning: invalid value encountered in scalar multiply
  0 * np.log(0)


nan